In [1]:
"""
Deep Learning Architectures Comparison on CIFAR-10
===================================================
Objectives: Implement and train three deep learning architectures 
(Simple NN, AlexNet, TinyVGG) on CIFAR-10 and compare performance.

Models: SimpleNN (baseline), AlexNet (modified for 32x32), TinyVGG (lightweight CNN)
AlexNet Paper: https://papers.nips.cc/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html
"""

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [2]:
# Define transformations for the dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load CIFAR-10 training dataset
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform
)

# Load CIFAR-10 test dataset
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform
)

# Create data loaders
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=64, shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=64, shuffle=False
)

# CIFAR-10 class names
classes = ('airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Number of classes: {len(classes)}")

100.0%


Training samples: 50000
Test samples: 10000
Number of classes: 10


In [3]:
class SimpleNN(nn.Module):
    """
    Simple fully connected neural network for image classification.
    Flattens the 32x32x3 input and passes through dense layers.
    """
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(32*32*3, 512)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Test the model architecture
simple_nn = SimpleNN().to(device)
print(simple_nn)

SimpleNN(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=3072, out_features=512, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=10, bias=True)
)


In [4]:
class AlexNetCIFAR(nn.Module):
    """
    Modified AlexNet architecture for CIFAR-10 (32x32 images).
    Original AlexNet was designed for 227x227 ImageNet images.
    
    Key adaptations:
    - Smaller kernel sizes (3x3 instead of 11x11, 5x5)
    - Fewer pooling layers to preserve spatial dimensions
    - Adjusted FC layer input size
    """
    def __init__(self):
        super(AlexNetCIFAR, self).__init__()
        self.features = nn.Sequential(
            # Conv1: 32x32x3 -> 32x32x64 -> 16x16x64 (after pooling)
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Conv2: 16x16x64 -> 16x16x192 -> 8x8x192 (after pooling)
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Conv3: 8x8x192 -> 8x8x384
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            # Conv4: 8x8x384 -> 8x8x256
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            
            # Conv5: 8x8x256 -> 8x8x256 -> 4x4x256 (after pooling)
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),  # Dropout as in original AlexNet
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Test the model architecture
alexnet = AlexNetCIFAR().to(device)
print(alexnet)

AlexNetCIFAR(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=4096, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p

In [5]:
class TinyVGG(nn.Module):
    """
    TinyVGG - A lightweight CNN architecture.
    Uses VGG-style 3x3 convolutions but with fewer layers.
    """
    def __init__(self):
        super(TinyVGG, self).__init__()
        
        self.features = nn.Sequential(
            # Block 1: 32x32x3 -> 32x32x32 -> 16x16x32
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Block 2: 16x16x32 -> 16x16x64 -> 8x8x64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Test the model architecture
tinyvgg = TinyVGG().to(device)
print(tinyvgg)

TinyVGG(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU(inplace=True)
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [6]:
# Define loss function (same for all models)
criterion = nn.CrossEntropyLoss()

# Learning rate for all models
learning_rate = 0.001
epochs = 10

print(f"Loss Function: CrossEntropyLoss")
print(f"Optimizer: Adam (lr={learning_rate})")
print(f"Epochs: {epochs}")

Loss Function: CrossEntropyLoss
Optimizer: Adam (lr=0.001)
Epochs: 10


In [7]:
def train_model(model, train_loader, criterion, optimizer, epochs=10, model_name="Model"):
    """
    Train a model and track metrics.
    
    Args:
        model: The neural network model
        train_loader: DataLoader for training data
        criterion: Loss function
        optimizer: Optimizer
        epochs: Number of training epochs
        model_name: Name for display purposes
    
    Returns:
        dict: Training history including losses and training time
    """
    model.train()
    history = {'loss': [], 'epoch_times': []}
    
    start_time = time.time()
    
    for epoch in range(epochs):
        epoch_start = time.time()
        running_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            # Zero the gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        epoch_loss = running_loss / len(train_loader)
        epoch_time = time.time() - epoch_start
        
        history['loss'].append(epoch_loss)
        history['epoch_times'].append(epoch_time)
        
        print(f"{model_name} - Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}, Time: {epoch_time:.2f}s")
    
    total_time = time.time() - start_time
    history['total_time'] = total_time
    
    print(f"\n{model_name} Training Complete!")
    print(f"Total Training Time: {total_time:.2f} seconds")
    
    return history

In [8]:
def evaluate_model(model, test_loader, model_name="Model"):
    """
    Evaluate a model on the test set and compute metrics.
    
    Args:
        model: The trained model
        test_loader: DataLoader for test data
        model_name: Name for display purposes
    
    Returns:
        dict: Evaluation metrics including accuracy and F1-score
    """
    model.eval()
    all_preds = []
    all_labels = []
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = 100 * correct / total
    f1 = f1_score(all_labels, all_preds, average='macro')
    
    print(f"\n{'='*50}")
    print(f"{model_name} Evaluation Results")
    print(f"{'='*50}")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"F1 Score (macro): {f1:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=classes))
    
    return {
        'accuracy': accuracy,
        'f1_score': f1,
        'predictions': all_preds,
        'labels': all_labels
    }

In [9]:
def count_parameters(model):
    """Count the total number of trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Compare model complexities
print("Model Parameter Comparison:")
print(f"{'Model':<20} {'Parameters':>15}")
print("-" * 35)
print(f"{'SimpleNN':<20} {count_parameters(simple_nn):>15,}")
print(f"{'AlexNetCIFAR':<20} {count_parameters(alexnet):>15,}")
print(f"{'TinyVGG':<20} {count_parameters(tinyvgg):>15,}")

Model Parameter Comparison:
Model                     Parameters
-----------------------------------
SimpleNN                   1,707,274
AlexNetCIFAR              35,855,178
TinyVGG                      591,274


In [ ]:
# Train all three models
print("="*60)
print("Training Simple Neural Network")
print("="*60)
simple_nn = SimpleNN().to(device)
optimizer_nn = optim.Adam(simple_nn.parameters(), lr=learning_rate)
history_nn = train_model(simple_nn, train_loader, criterion, optimizer_nn, epochs, "SimpleNN")

print("\n" + "="*60)
print("Training AlexNet (Modified for CIFAR-10)")
print("="*60)
alexnet = AlexNetCIFAR().to(device)
optimizer_alex = optim.Adam(alexnet.parameters(), lr=learning_rate)
history_alex = train_model(alexnet, train_loader, criterion, optimizer_alex, epochs, "AlexNet")

print("\n" + "="*60)
print("Training TinyVGG")
print("="*60)
tinyvgg = TinyVGG().to(device)
optimizer_vgg = optim.Adam(tinyvgg.parameters(), lr=learning_rate)
history_vgg = train_model(tinyvgg, train_loader, criterion, optimizer_vgg, epochs, "TinyVGG")

In [ ]:
# Evaluate all three models
results_nn = evaluate_model(simple_nn, test_loader, "SimpleNN")
results_alex = evaluate_model(alexnet, test_loader, "AlexNet")
results_vgg = evaluate_model(tinyvgg, test_loader, "TinyVGG")

In [ ]:
# Comparison Table and Visualizations
print("\n" + "="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)

comparison_data = {
    'Model': ['SimpleNN', 'AlexNet', 'TinyVGG'],
    'Accuracy (%)': [results_nn['accuracy'], results_alex['accuracy'], results_vgg['accuracy']],
    'F1-Score': [results_nn['f1_score'], results_alex['f1_score'], results_vgg['f1_score']],
    'Training Time (s)': [history_nn['total_time'], history_alex['total_time'], history_vgg['total_time']],
    'Parameters': [count_parameters(simple_nn), count_parameters(alexnet), count_parameters(tinyvgg)]
}

print(f"\n{'Model':<15} {'Accuracy':<12} {'F1-Score':<12} {'Time (s)':<12} {'Parameters':<15}")
print("-" * 70)
for i in range(3):
    print(f"{comparison_data['Model'][i]:<15} "
          f"{comparison_data['Accuracy (%)'][i]:<12.2f} "
          f"{comparison_data['F1-Score'][i]:<12.4f} "
          f"{comparison_data['Training Time (s)'][i]:<12.2f} "
          f"{comparison_data['Parameters'][i]:<15,}")

# Training Loss, Accuracy and F1 Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history_nn['loss'], label='SimpleNN', marker='o')
axes[0].plot(history_alex['loss'], label='AlexNet', marker='s')
axes[0].plot(history_vgg['loss'], label='TinyVGG', marker='^')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()
axes[0].grid(True)

models = ['SimpleNN', 'AlexNet', 'TinyVGG']
accuracies = [results_nn['accuracy'], results_alex['accuracy'], results_vgg['accuracy']]
colors = ['#3498db', '#e74c3c', '#2ecc71']
axes[1].bar(models, accuracies, color=colors)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Test Accuracy Comparison')
axes[1].set_ylim([0, 100])
for i, v in enumerate(accuracies):
    axes[1].text(i, v + 1, f'{v:.2f}%', ha='center')

f1_scores = [results_nn['f1_score'], results_alex['f1_score'], results_vgg['f1_score']]
axes[2].bar(models, f1_scores, color=colors)
axes[2].set_ylabel('F1-Score')
axes[2].set_title('F1-Score Comparison (Macro)')
axes[2].set_ylim([0, 1])
for i, v in enumerate(f1_scores):
    axes[2].text(i, v + 0.02, f'{v:.4f}', ha='center')

plt.tight_layout()
plt.show()

# Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, results, name in zip(axes, [results_nn, results_alex, results_vgg], ['SimpleNN', 'AlexNet', 'TinyVGG']):
    cm = confusion_matrix(results['labels'], results['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'{name} Confusion Matrix')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

## Discussion

### Model Performance Analysis

#### 1. Simple Neural Network (NN)
- **Lowest performance** as expected
- **Why?** Ignores spatial structure of images by flattening all pixels
- No parameter sharing (unlike CNNs)
- Cannot learn local patterns (edges, textures, shapes)
- Acts as a baseline for comparison

#### 2. TinyVGG
- **Medium performance** - good balance of simplicity and accuracy
- Uses convolutional layers to learn spatial features
- Fewer parameters than AlexNet → faster training
- Two conv blocks capture low and mid-level features
- Good choice when computational resources are limited

#### 3. AlexNet (Modified)
- **Best performance** among the three models
- Deeper architecture learns more complex features
- Dropout regularization prevents overfitting
- More filters capture richer representations
- Trade-off: Longest training time and most parameters

### AlexNet Improvements Over Predecessors (LeNet, etc.)

| Innovation | Description | Impact |
|------------|-------------|--------|
| **ReLU Activation** | Non-saturating activation function | 6x faster training than tanh |
| **Dropout** | Randomly zeroes neurons during training | Reduces overfitting |
| **Data Augmentation** | Image translations, reflections | Better generalization |
| **Local Response Normalization** | Lateral inhibition (like biology) | Improved accuracy |
| **Overlapping Pooling** | Stride < kernel size | Reduced overfitting |
| **GPU Training** | Parallelized across 2 GPUs | Enabled training on large datasets |

### Why F1-Score?
- **Accuracy alone can be misleading** when classes are imbalanced
- F1-Score balances **Precision** (false positives) and **Recall** (false negatives)
- Using `macro` averaging treats all classes equally
- More robust metric for multi-class classification

### Trade-offs Observed
| Aspect | SimpleNN | TinyVGG | AlexNet |
|--------|----------|---------|---------|
| Accuracy | Low | Medium | High |
| Training Time | Fast | Medium | Slow |
| Parameters | Low | Medium | High |
| Complexity | Simple | Moderate | Complex |

### Recommendations
- **For quick prototyping**: TinyVGG offers good balance
- **For best accuracy**: AlexNet (or deeper models)
- **Never use**: Fully connected NN for image classification (no spatial awareness)

## Conclusion

In this project, we successfully implemented and compared three neural network architectures on the CIFAR-10 dataset:

### Key Findings:

1. **Convolutional networks (AlexNet, TinyVGG) significantly outperform fully connected networks** for image classification tasks due to their ability to learn spatial hierarchies.

2. **AlexNet achieved the highest accuracy and F1-score**, demonstrating the effectiveness of deeper architectures with regularization techniques like Dropout.

3. **TinyVGG provides a good trade-off** between complexity and performance, making it suitable for resource-constrained environments.

4. **The Simple NN serves as a baseline** showing why spatial feature extraction (convolutions) is essential for image tasks.

### Technical Choices Justification:

| Choice | Reason |
|--------|--------|
| **CrossEntropyLoss** | Standard for multi-class classification; combines softmax and NLL |
| **Adam Optimizer** | Adaptive learning rate; fast convergence; robust to hyperparameters |
| **F1-Score (macro)** | Balanced metric for 10-class classification; handles class variations |
| **Batch Size 64** | Good trade-off between training speed and gradient quality |
| **10 Epochs** | Sufficient for convergence demonstration; prevents overfitting |

### Future Improvements:
- Add data augmentation (random crops, flips) for better generalization
- Implement learning rate scheduling
- Try modern architectures (ResNet, EfficientNet)
- Add batch normalization for faster training
- Experiment with different optimizers (SGD with momentum)
